In [35]:
import pandas as pd

def clean_excel(file_name):
    # Read raw file
    df = pd.read_excel(file_name, header=None)

    # Set second row as header
    df.columns = df.iloc[1]

    # Remove title row and header row
    df = df.iloc[2:].reset_index(drop=True)

    # Remove header name
    df.columns.name = None

    # Clean column names
    df.columns = [str(col).strip() for col in df.columns]

    # Clean only string values
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)

    return df

In [36]:
companies = clean_excel("companies.xlsx")
companies.head()

,id,company_logo,company_name,chart_link,about_company,website,nse_profile,bse_profile,face_value,book_value,roce_percentage,roe_percentage
0,ABB,https://mkt.in/static/mkt-icons/nifty100/ABB.png,Abbott India Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,Abbott India Ltd is one of the leading multina...,https://www.abbott.co.in/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/abb...,10.0,1657.0,46.0,34.90
1,ADANIENSOL,https://m.economictimes.com/thumb/msid-1173715...,Adani Energy Solutions Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"AESL, part of the Adani portfolio, is a multid...",https://www.adanienergysolutions.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10.0,175.0,9.0,8.59
2,ADANIENT,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Enterprises Ltd,https://in.tradingview.com/chart/?symbol=ADANIENT,Adani Enterprises Ltd is an Indian multination...,https://www.adanienterprises.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,1.0,363.0,11.6,13.64
3,ADANIGREEN,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Green Energy Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"Adani Green Energy Limited, incorporated in 20...",http://www.adanigreenenergy.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10.0,67.0,96.5,14.70
4,ADANIPORTS,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Ports & Special Economic Zone Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,Adani Ports & Special Economic Zone is in the ...,http://www.adaniports.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,2.0,265.0,12.9,18.10


In [37]:
companies = clean_excel("companies.xlsx")
sectors = clean_excel("sectors.xlsx")
peer_groups = clean_excel("peer_groups.xlsx")
profit_loss = clean_excel("profitandloss.xlsx")
balance_sheet = clean_excel("balancesheet.xlsx")
cash_flow = clean_excel("cashflow.xlsx")
financial_ratios = clean_excel("financial_ratios.xlsx")
market_cap = clean_excel("market_cap.xlsx")
stock_prices = clean_excel("stock_prices.xlsx")
analysis = clean_excel("analysis.xlsx")
pros_cons = clean_excel("prosandcons.xlsx")
documents = clean_excel("documents.xlsx")

print("✅ All datasets cleaned successfully!")

✅ All datasets cleaned successfully!


In [38]:
datasets = {
    "companies": companies,
    "sectors": sectors,
    "peer_groups": peer_groups,
    "profit_loss": profit_loss,
    "balance_sheet": balance_sheet,
    "cash_flow": cash_flow,
    "financial_ratios": financial_ratios,
    "market_cap": market_cap,
    "stock_prices": stock_prices,
    "analysis": analysis,
    "pros_cons": pros_cons,
    "documents": documents
}

In [39]:
import re

def standardize_columns(df):
    cols = []

    for c in df.columns:
        c = str(c).strip().lower()
        c = c.replace("%", "pct")
        c = c.replace("&", "and")
        c = c.replace("/", "_")
        c = c.replace("-", "_")
        c = c.replace(" ", "_")
        c = re.sub(r'[^a-zA-Z0-9_]', '', c)
        c = re.sub(r'_+', '_', c)
        cols.append(c)

    df.columns = cols
    return df


for name in datasets:
    datasets[name] = standardize_columns(datasets[name])

In [40]:
datasets["companies"].dtypes.head(10)

id                object
company_logo      object
company_name      object
chart_link        object
about_company     object
website           object
nse_profile       object
bse_profile       object
face_value       float64
book_value       float64
dtype: object

In [41]:
def convert_numeric(df):
    for col in df.columns:
        if df[col].dtype == object:

            cleaned = (
                df[col]
                .astype(str)
                .str.replace(",", "", regex=False)
                .str.replace("%", "", regex=False)
                .str.replace("₹", "", regex=False)
                .str.strip()
            )

            try:
                df[col] = pd.to_numeric(cleaned)
            except (ValueError, TypeError):
                df[col] = cleaned

    return df

In [42]:
import sqlite3

In [43]:
for name, df in datasets.items():
    try:
        df.to_sql(
            name,
            sqlite3.connect(":memory:"),
            if_exists="replace",
            index=False
        )
        print("✅", name)

    except Exception as e:
        print("❌", name, "---->", e)

✅ companies
✅ sectors
✅ peer_groups
✅ profit_loss
✅ balance_sheet
✅ cash_flow
❌ financial_ratios ----> duplicate column name: 0
✅ market_cap
❌ stock_prices ----> duplicate column name: 19643
✅ analysis
✅ pros_cons
✅ documents


In [44]:
financial_ratios.columns[financial_ratios.columns.duplicated()]



Index(['0'], dtype='object')

In [45]:
stock_prices.columns[stock_prices.columns.duplicated()]

Index(['19643'], dtype='object')

In [46]:
for name, df in datasets.items():
    try:
        df.to_sql(
            name,
            sqlite3.connect(":memory:"),
            if_exists="replace",
            index=False
        )
        print("✅", name)

    except Exception as e:
        print("❌", name, "---->", e)

✅ companies
✅ sectors
✅ peer_groups
✅ profit_loss
✅ balance_sheet
✅ cash_flow
❌ financial_ratios ----> duplicate column name: 0
✅ market_cap
❌ stock_prices ----> duplicate column name: 19643
✅ analysis
✅ pros_cons
✅ documents


In [47]:
financial_ratios.columns.tolist()[:10]

['1', 'abb', 'dec_2012', '877', '12', '2241', '0', 'nan', '18225', '42']

In [48]:
stock_prices.columns.tolist()[:10]

['1',
 'abb',
 '2020_01_01',
 '195861',
 '215053',
 '183528',
 '19643',
 '26184368',
 '19643']

In [49]:
financial_ratios = financial_ratios.loc[:, financial_ratios.columns.astype(str) != "0"]

stock_prices = stock_prices.loc[:, stock_prices.columns.astype(str) != "19643"]

In [50]:
financial_ratios = financial_ratios.loc[:, ~financial_ratios.columns.duplicated()]

stock_prices = stock_prices.loc[:, ~stock_prices.columns.duplicated()]

In [51]:
datasets["financial_ratios"] = financial_ratios
datasets["stock_prices"] = stock_prices

In [52]:
for name in ["financial_ratios", "stock_prices"]:
    print(name)
    print(datasets[name].columns[datasets[name].columns.duplicated()])

financial_ratios
Index([], dtype='object')
stock_prices
Index([], dtype='object')


In [53]:
for name, df in datasets.items():
    try:
        df.to_sql(
            name,
            sqlite3.connect(":memory:"),
            if_exists="replace",
            index=False
        )
        print("✅", name)

    except Exception as e:
        print("❌", name, "---->", e)

✅ companies
✅ sectors
✅ peer_groups
✅ profit_loss
✅ balance_sheet
✅ cash_flow
✅ financial_ratios
✅ market_cap
✅ stock_prices
✅ analysis
✅ pros_cons
✅ documents


In [54]:
import sqlite3

conn = sqlite3.connect("nifty100.db")

for table, df in datasets.items():
    df.to_sql(
        table,
        conn,
        if_exists="replace",
        index=False
    )

conn.commit()

print("✅ Nifty100 database created successfully")

✅ Nifty100 database created successfully


In [55]:
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

tables

,name
0,master_company_data
1,companies
2,sectors
3,peer_groups
4,profit_loss
5,balance_sheet
6,cash_flow
7,financial_ratios
8,market_cap
9,stock_prices


In [56]:
print("COMPANIES")
print(companies.columns.tolist())

print("\nFINANCIAL RATIOS")
print(financial_ratios.columns.tolist())

print("\nMARKET CAP")
print(market_cap.columns.tolist())

COMPANIES
['id', 'company_logo', 'company_name', 'chart_link', 'about_company', 'website', 'nse_profile', 'bse_profile', 'face_value', 'book_value', 'roce_percentage', 'roe_percentage']

FINANCIAL RATIOS
['1', 'abb', 'dec_2012', '877', '12', '2241', 'nan', '18225', '42', '59', '68', '3081', '25', '101']

MARKET CAP
['1', 'abb', '2019', '84431298', '81441048', '1929', '1426', '2284', '084']


In [57]:
financial_ratios.head()

,1,abb,dec_2012,877,12,2241,nan,18225,42,59,68,3081,25,101
0,2,ABB,Mar 2014,8.70,12.0,25.13,NaN,1.9982,11.0,144.0,93.0,3.7524,25.0,155.0
1,3,ABB,Mar 2014,8.70,12.0,25.13,NaN,1.9982,0.0,0.0,93.0,3.7524,25.0,0.0
2,4,ABB,Mar 2015,10.00,14.0,24.44,NaN,1.6659,28.0,187.0,108.0,4.4619,29.0,215.0
3,5,ABB,Mar 2015,10.00,14.0,24.44,NaN,1.6659,-1899.0,1864.0,108.0,4.4619,29.0,-35.0
4,6,ABB,Mar 2016,9.76,14.0,21.34,138.3333,1.6176,172.0,77.0,120.0,5.6905,29.0,249.0


In [58]:
market_cap.head()

,1,abb,2019,84431298,81441048,1929,1426,2284,084
0,2,ABB,2020,923674.33,1120679.72,67.54,13.03,24.93,3.90
1,3,ABB,2021,1020674.09,1166450.81,62.63,6.67,7.58,1.58
2,4,ABB,2022,1185994.82,1158637.42,18.68,4.75,6.12,3.35
3,5,ABB,2023,1161513.28,1103083.77,58.34,12.72,5.70,1.05
4,6,ABB,2024,1536236.24,1619349.16,75.58,4.48,12.63,0.96


In [59]:
financial_ratios.shape

(1183, 14)

In [60]:
financial_ratios.shape
financial_ratios.columns.tolist()

['1',
 'abb',
 'dec_2012',
 '877',
 '12',
 '2241',
 'nan',
 '18225',
 '42',
 '59',
 '68',
 '3081',
 '25',
 '101']

In [61]:
financial_ratios = clean_excel("financial_ratios.xlsx")

In [62]:
%whos

Variable              Type          Data/Info
---------------------------------------------
analysis              DataFrame     Shape: (20, 6)
balance_sheet         DataFrame     Shape: (1312, 13)
cash_flow             DataFrame     Shape: (1187, 7)
clean_excel           function      <function clean_excel at 0x0000028BE1C40460>
companies             DataFrame     Shape: (92, 12)
conn                  Connection    <sqlite3.Connection object at 0x0000028BDF54C310>
convert_numeric       function      <function convert_numeric at 0x0000028BDF3277F0>
datasets              dict          n=12
df                    DataFrame     Shape: (1585, 4)
documents             DataFrame     Shape: (1585, 4)
financial_ratios      DataFrame     Shape: (1183, 16)
latest_pl             DataFrame     Shape: (100, 15)
market_cap            DataFrame     Shape: (551, 9)
master                DataFrame     Shape: (104, 17)
name                  str           documents
pd                    module        <modu

In [63]:
companies = clean_excel("companies.xlsx")
sectors = clean_excel("sectors.xlsx")
peer_groups = clean_excel("peer_groups.xlsx")
profit_loss = clean_excel("profitandloss.xlsx")
balance_sheet = clean_excel("balancesheet.xlsx")
cash_flow = clean_excel("cashflow.xlsx")
financial_ratios = clean_excel("financial_ratios.xlsx")
market_cap = clean_excel("market_cap.xlsx")
stock_prices = clean_excel("stock_prices.xlsx")
analysis = clean_excel("analysis.xlsx")
pros_cons = clean_excel("prosandcons.xlsx")
documents = clean_excel("documents.xlsx")

In [64]:
financial_ratios.head()
financial_ratios.shape

(1183, 16)

In [65]:
for name, df in datasets.items():
    print("="*70)
    print(name.upper())
    print(df.columns.tolist())

COMPANIES
['id', 'company_logo', 'company_name', 'chart_link', 'about_company', 'website', 'nse_profile', 'bse_profile', 'face_value', 'book_value', 'roce_percentage', 'roe_percentage']
SECTORS
['1', 'abb', 'industrials', 'capital_goods', '081', 'large_cap']
PEER_GROUPS
['2', 'private_banks', 'icicibank', 'false']
PROFIT_LOSS
['id', 'company_id', 'year', 'sales', 'expenses', 'operating_profit', 'opm_percentage', 'other_income', 'interest', 'depreciation', 'profit_before_tax', 'tax_percentage', 'net_profit', 'eps', 'dividend_payout']
BALANCE_SHEET
['id', 'company_id', 'year', 'equity_capital', 'reserves', 'borrowings', 'other_liabilities', 'total_liabilities', 'fixed_assets', 'cwip', 'investments', 'other_asset', 'total_assets']
CASH_FLOW
['id', 'company_id', 'year', 'operating_activity', 'investing_activity', 'financing_activity', 'net_cash_flow']
FINANCIAL_RATIOS
['1', 'abb', 'dec_2012', '877', '12', '2241', 'nan', '18225', '42', '59', '68', '3081', '25', '101']
MARKET_CAP
['1', 'abb'

In [66]:
master = (
    companies
    .merge(analysis, left_on="id", right_on="company_id", how="left")
)

master = master.drop(columns=["company_id"], errors="ignore")

master.head()

,id_x,company_logo,company_name,chart_link,about_company,website,nse_profile,bse_profile,face_value,book_value,roce_percentage,roe_percentage,id_y,compounded_sales_growth,compounded_profit_growth,stock_price_cagr,roe
0,ABB,https://mkt.in/static/mkt-icons/nifty100/ABB.png,Abbott India Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,Abbott India Ltd is one of the leading multina...,https://www.abbott.co.in/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/abb...,10.0,1657.0,46.0,34.90,NaN,NaN,NaN,NaN,NaN
1,ADANIENSOL,https://m.economictimes.com/thumb/msid-1173715...,Adani Energy Solutions Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"AESL, part of the Adani portfolio, is a multid...",https://www.adanienergysolutions.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10.0,175.0,9.0,8.59,NaN,NaN,NaN,NaN,NaN
2,ADANIENT,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Enterprises Ltd,https://in.tradingview.com/chart/?symbol=ADANIENT,Adani Enterprises Ltd is an Indian multination...,https://www.adanienterprises.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,1.0,363.0,11.6,13.64,NaN,NaN,NaN,NaN,NaN
3,ADANIGREEN,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Green Energy Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"Adani Green Energy Limited, incorporated in 20...",http://www.adanigreenenergy.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10.0,67.0,96.5,14.70,NaN,NaN,NaN,NaN,NaN
4,ADANIPORTS,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Ports & Special Economic Zone Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,Adani Ports & Special Economic Zone is in the ...,http://www.adaniports.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,2.0,265.0,12.9,18.10,NaN,NaN,NaN,NaN,NaN


In [67]:
master = (
    companies
    .merge(analysis, left_on="id", right_on="company_id", how="left")
)

master = master.drop(columns=["company_id"], errors="ignore")

master.head()

,id_x,company_logo,company_name,chart_link,about_company,website,nse_profile,bse_profile,face_value,book_value,roce_percentage,roe_percentage,id_y,compounded_sales_growth,compounded_profit_growth,stock_price_cagr,roe
0,ABB,https://mkt.in/static/mkt-icons/nifty100/ABB.png,Abbott India Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,Abbott India Ltd is one of the leading multina...,https://www.abbott.co.in/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/abb...,10.0,1657.0,46.0,34.90,NaN,NaN,NaN,NaN,NaN
1,ADANIENSOL,https://m.economictimes.com/thumb/msid-1173715...,Adani Energy Solutions Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"AESL, part of the Adani portfolio, is a multid...",https://www.adanienergysolutions.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10.0,175.0,9.0,8.59,NaN,NaN,NaN,NaN,NaN
2,ADANIENT,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Enterprises Ltd,https://in.tradingview.com/chart/?symbol=ADANIENT,Adani Enterprises Ltd is an Indian multination...,https://www.adanienterprises.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,1.0,363.0,11.6,13.64,NaN,NaN,NaN,NaN,NaN
3,ADANIGREEN,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Green Energy Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"Adani Green Energy Limited, incorporated in 20...",http://www.adanigreenenergy.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10.0,67.0,96.5,14.70,NaN,NaN,NaN,NaN,NaN
4,ADANIPORTS,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Ports & Special Economic Zone Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,Adani Ports & Special Economic Zone is in the ...,http://www.adaniports.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,2.0,265.0,12.9,18.10,NaN,NaN,NaN,NaN,NaN


In [68]:
latest_pl = (
    profit_loss
    .sort_values("year")
    .groupby("company_id")
    .last()
    .reset_index()
)

master = master.merge(
    latest_pl,
    left_on="id",
    right_on="company_id",
    how="left",
    suffixes=("", "_pl")
)

master.drop(columns=["company_id"], inplace=True)

master.head()

KeyError: 'id'

In [ ]:
print(master.columns.tolist())

['id_x', 'company_logo', 'company_name', 'chart_link', 'about_company', 'website', 'nse_profile', 'bse_profile', 'face_value', 'book_value', 'roce_percentage', 'roe_percentage', 'id_y', 'compounded_sales_growth', 'compounded_profit_growth', 'stock_price_cagr', 'roe']


In [ ]:
print(companies.columns.tolist())

['id', 'company_logo', 'company_name', 'chart_link', 'about_company', 'website', 'nse_profile', 'bse_profile', 'face_value', 'book_value', 'roce_percentage', 'roe_percentage']


In [ ]:
master = master.rename(columns={"id_x": "company_id"})
master = master.drop(columns=["id_y"], errors="ignore")

master.columns.tolist()

['company_id',
 'company_logo',
 'company_name',
 'chart_link',
 'about_company',
 'website',
 'nse_profile',
 'bse_profile',
 'face_value',
 'book_value',
 'roce_percentage',
 'roe_percentage',
 'compounded_sales_growth',
 'compounded_profit_growth',
 'stock_price_cagr',
 'roe']

In [ ]:
latest_pl = (
    profit_loss
    .sort_values("year")
    .groupby("company_id", as_index=False)
    .last()
)

master = master.merge(
    latest_pl,
    on="company_id",
    how="left",
    suffixes=("", "_pl")
)

In [ ]:
latest_bs = (
    balance_sheet
    .sort_values("year")
    .groupby("company_id", as_index=False)
    .last()
)

master = master.merge(
    latest_bs,
    on="company_id",
    how="left",
    suffixes=("", "_bs")
)

In [ ]:
latest_cf = (
    cash_flow
    .sort_values("year")
    .groupby("company_id", as_index=False)
    .last()
)

master = master.merge(
    latest_cf,
    on="company_id",
    how="left",
    suffixes=("", "_cf")
)

In [ ]:
master = master.merge(
    pros_cons,
    on="company_id",
    how="left",
    suffixes=("", "_pc")
)

In [ ]:
latest_doc = (
    documents
    .sort_values("Year")
    .groupby("company_id", as_index=False)
    .last()
)

master = master.merge(
    latest_doc,
    on="company_id",
    how="left",
    suffixes=("", "_doc")
)

In [ ]:
print(documents.columns.tolist())

['id', 'company_id', 'Year', 'Annual_Report']


In [ ]:
print(master.shape)
master.head()

(144, 54)


,company_id,company_logo,company_name,chart_link,about_company,website,nse_profile,bse_profile,face_value,book_value,...,operating_activity,investing_activity,financing_activity,net_cash_flow,id_pc,pros,cons,id_doc,Year,Annual_Report
0,ABB,https://mkt.in/static/mkt-icons/nifty100/ABB.png,Abbott India Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,Abbott India Ltd is one of the leading multina...,https://www.abbott.co.in/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/abb...,10.0,1657.0,...,6038.0,-4943.0,-543.0,551.0,NaN,NaN,NaN,1.0,2024.0,https://www.bseindia.com/xml-data/corpfiling/A...
1,ADANIENSOL,https://m.economictimes.com/thumb/msid-1173715...,Adani Energy Solutions Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"AESL, part of the Adani portfolio, is a multid...",https://www.adanienergysolutions.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10.0,175.0,...,6038.0,-4943.0,-543.0,551.0,NaN,NaN,NaN,17.0,2024.0,https://www.bseindia.com/xml-data/corpfiling/A...
2,ADANIENT,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Enterprises Ltd,https://in.tradingview.com/chart/?symbol=ADANIENT,Adani Enterprises Ltd is an Indian multination...,https://www.adanienterprises.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,1.0,363.0,...,10312.0,-18767.0,8879.0,424.0,NaN,NaN,NaN,33.0,2024.0,https://www.bseindia.com/xml-data/corpfiling/A...
3,ADANIGREEN,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Green Energy Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"Adani Green Energy Limited, incorporated in 20...",http://www.adanigreenenergy.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10.0,67.0,...,7713.0,-21060.0,13953.0,606.0,NaN,NaN,NaN,49.0,2024.0,https://www.bseindia.com/xml-data/corpfiling/A...
4,ADANIPORTS,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Ports & Special Economic Zone Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,Adani Ports & Special Economic Zone is in the ...,http://www.adaniports.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,2.0,265.0,...,15018.0,-6768.0,-7800.0,450.0,NaN,NaN,NaN,65.0,2024.0,https://www.bseindia.com/xml-data/corpfiling/A...


In [ ]:
import pandas as pd
import numpy as np

numeric_cols = [
    "market_cap",
    "sales",
    "net_profit",
    "eps",
    "roe_percentage",
    "roce_percentage",
    "book_value",
    "stock_price_cagr",
    "compounded_sales_growth",
    "compounded_profit_growth"
]

for col in numeric_cols:
    if col in master.columns:
        master[col] = pd.to_numeric(master[col], errors="coerce")

In [ ]:
def screener(
    df,
    min_roe=None,
    min_roce=None,
    min_sales=None,
    min_profit=None
):
    result = df.copy()

    if min_roe is not None and "roe_percentage" in result.columns:
        result = result[result["roe_percentage"] >= min_roe]

    if min_roce is not None and "roce_percentage" in result.columns:
        result = result[result["roce_percentage"] >= min_roce]

    if min_sales is not None and "sales" in result.columns:
        result = result[result["sales"] >= min_sales]

    if min_profit is not None and "net_profit" in result.columns:
        result = result[result["net_profit"] >= min_profit]

    return result.reset_index(drop=True)

In [ ]:
screened = screener(
    master,
    min_roe=15,
    min_roce=15
)

print(screened.shape)

screened[[
    "company_name",
    "roe_percentage",
    "roce_percentage"
]].head(20)

(43, 54)


,company_name,roe_percentage,roce_percentage
0,Abbott India Ltd,34.90,46.00
1,Adani Power Ltd,57.10,32.20
2,Asian Paints\nIndian Multi-National Paint and ...,31.45,41.75
3,Adani Total Gas Ltd,20.50,21.20
4,Bajaj Auto Ltd,26.50,33.50
5,Bharat Electronics Ltd,26.30,34.60
6,Bosch Ltd,16.00,20.60
7,Bharat Petroleum Corporation Ltd,41.90,32.10
8,Britannia Industries Ltd,57.10,48.90
9,Cipla Ltd,16.80,22.80


In [ ]:
ranking = master.copy()

ranking["Overall Score"] = (
    ranking["roe_percentage"].fillna(0)
    + ranking["roce_percentage"].fillna(0)
    + ranking["compounded_profit_growth"].fillna(0)
)

ranking = ranking.sort_values(
    "Overall Score",
    ascending=False
)

ranking[[
    "company_name",
    "Overall Score"
]].head(20)

,company_name,Overall Score
88,Nestle India Ltd,321.04
80,Life Insurance Corporation of India,127.40
25,Coal India Ltd,115.60
3,Adani Green Energy Ltd,111.20
21,Britannia Industries Ltd,106.00
72,Indian Railway Catering & Tourism Corporation Ltd,94.20
5,Adani Power Ltd,89.30
0,Abbott India Ltd,80.90
20,Bharat Petroleum Corporation Ltd,74.00
8,Asian Paints\nIndian Multi-National Paint and ...,73.20


In [ ]:
top10 = ranking.head(10)

top10

,company_id,company_logo,company_name,chart_link,about_company,website,nse_profile,bse_profile,face_value,book_value,...,investing_activity,financing_activity,net_cash_flow,id_pc,pros,cons,id_doc,Year,Annual_Report,Overall Score
88,NESTLEIND,https://mkt.in/static/mkt-icons/nifty100/NESTL...,Nestle India Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,Nestle India Limited is the Indian subsidiary ...,https://www.nestle.in/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/nes...,1.0,41.0,...,-1237.0,-3135.0,-198.0,NaN,NaN,NaN,1042.0,2024.0,https://www.bseindia.com/xml-data/corpfiling/A...,321.04
80,LICI,https://mkt.in/static/mkt-icons/nifty100/LICI.png,Life Insurance Corporation of India,https://in.tradingview.com/chart/?symbol=NSE%3...,Life Insurance Corporation of India is an Indi...,https://www.licindia.in/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/lif...,10.0,154.0,...,-25269.0,-4427.0,-3574.0,NaN,NaN,NaN,914.0,2024.0,https://www.bseindia.com/xml-data/corpfiling/A...,127.40
25,COALINDIA,https://mkt.in/static/mkt-icons/nifty100/COALI...,Coal India Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,Coal India Ltd is mainly engaged in mining and...,http://www.coalindia.in/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/coa...,10.0,156.0,...,-4486.0,-13899.0,-282.0,NaN,NaN,NaN,401.0,2024.0,https://www.bseindia.com/xml-data/corpfiling/A...,115.60
3,ADANIGREEN,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Green Energy Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"Adani Green Energy Limited, incorporated in 20...",http://www.adanigreenenergy.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10.0,67.0,...,-21060.0,13953.0,606.0,NaN,NaN,NaN,49.0,2024.0,https://www.bseindia.com/xml-data/corpfiling/A...,111.20
21,BRITANNIA,https://mkt.in/static/mkt-icons/nifty100/BRITA...,Britannia Industries Ltd,https://in.tradingview.com/chart/qGsydD2w/?sym...,Britannia Industries is one of Indias leading ...,https://www.britannia.co.in/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/bri...,1.0,133.0,...,477.0,-2830.0,219.0,NaN,NaN,NaN,337.0,2024.0,https://www.bseindia.com/xml-data/corpfiling/A...,106.00
72,IRCTC,https://mkt.in/static/mkt-icons/nifty100/IRCTC...,Indian Railway Catering & Tourism Corporation Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"Incorporated in 1999, IRCTC is a Mini Ratna (C...",http://www.irctc.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ind...,2.0,44.0,...,-215.0,-404.0,262.0,NaN,NaN,NaN,786.0,2024.0,https://www.bseindia.com/xml-data/corpfiling/A...,94.20
5,ADANIPOWER,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Power Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"Adani Power (APL), a part of the diversified A...",http://www.adani.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10.0,145.0,...,3481.0,-16864.0,787.0,NaN,NaN,NaN,81.0,2024.0,https://www.bseindia.com/bseplus/AnnualReport/...,89.30
0,ABB,https://mkt.in/static/mkt-icons/nifty100/ABB.png,Abbott India Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,Abbott India Ltd is one of the leading multina...,https://www.abbott.co.in/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/abb...,10.0,1657.0,...,-4943.0,-543.0,551.0,NaN,NaN,NaN,1.0,2024.0,https://www.bseindia.com/xml-data/corpfiling/A...,80.90
20,BPCL,https://mkt.in/static/mkt-icons/nifty100/BPCL.png,Bharat Petroleum Corporation Ltd,https://in.tradingview.com/chart/qGsydD2w/?sym...,Bharat Petroleum Corporation is a public secto...,https://www.bharatpetroleum.in/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/bha...,10.0,178

In [ ]:
screened.to_excel(
    "screened_companies.xlsx",
    index=False
)

ranking.to_excel(
    "company_ranking.xlsx",
    index=False
)

print("✅ Screener files exported")

✅ Screener files exported


In [ ]:
peer_compare = master[[
    "company_name",
    "sales",
    "net_profit",
    "roe_percentage",
    "roce_percentage",
    "eps",
    "book_value"
]].copy()

peer_compare.head()

,company_name,sales,net_profit,roe_percentage,roce_percentage,eps,book_value
0,Abbott India Ltd,6066,1285,34.90,46.0,605.0,1657.0
1,Adani Energy Solutions Ltd,22099,589,8.59,9.0,6.0,175.0
2,Adani Enterprises Ltd,102311,6086,13.64,11.6,49.0,363.0
3,Adani Green Energy Ltd,10782,1928,14.70,96.5,9.0,67.0
4,Adani Ports & Special Economic Zone Ltd,28443,9743,18.10,12.9,45.0,265.0


In [ ]:
companies_to_compare = [
    "ABB",
    "Adani Enterprises",
    "Asian Paints"
]

comparison = peer_compare[
    peer_compare["company_name"].isin(companies_to_compare)
]

comparison

,company_name,sales,net_profit,roe_percentage,roce_percentage,eps,book_value


In [ ]:
peer_groups = clean_excel("peer_groups.xlsx")

peer_groups.columns = [
    "id",
    "peer_group",
    "company_id",
    "is_leader"
]

peer_groups.head()

,id,peer_group,company_id,is_leader
0,2,Private Banks,ICICIBANK,False
1,3,Private Banks,AXISBANK,False
2,4,Private Banks,KOTAKBANK,False
3,5,Private Banks,INDUSINDBK,False
4,6,Public Sector Banks,SBIN,True


In [ ]:
peer_groups.to_excel(
    "peer_groups.xlsx",
    index=False
)

In [ ]:
import sqlite3

conn = sqlite3.connect("nifty100.db")

peer_groups.to_sql(
    "peer_groups",
    conn,
    if_exists="replace",
    index=False
)

conn.close()

print("✅ peer_groups table updated.")

✅ peer_groups table updated.


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("nifty100.db")

df = pd.read_sql("SELECT * FROM peer_groups LIMIT 5", conn)

conn.close()

print(df.columns.tolist())
print(df.head())

['id', 'peer_group', 'company_id', 'is_leader']
   id           peer_group  company_id  is_leader
0   2        Private Banks   ICICIBANK          0
1   3        Private Banks    AXISBANK          0
2   4        Private Banks   KOTAKBANK          0
3   5        Private Banks  INDUSINDBK          0
4   6  Public Sector Banks        SBIN          1


In [ ]:
import pandas as pd

companies = pd.read_excel("companies.xlsx")
analysis = pd.read_excel("analysis.xlsx")
profit_loss = pd.read_excel("profitandloss.xlsx")
peer_groups = pd.read_excel("peer_groups.xlsx")

In [ ]:
latest_pl = (
    profit_loss
    .sort_values("year")
    .groupby("company_id")
    .last()
    .reset_index()
)

In [ ]:
master = companies.merge(
    analysis,
    left_on="id",
    right_on="company_id",
    how="left"
)

In [ ]:
master = master.merge(
    latest_pl,
    left_on="id",
    right_on="company_id",
    how="left",
    suffixes=("", "_pl")
)

In [ ]:
print(master.columns.tolist())

['id_x', 'company_logo', 'company_name', 'chart_link', 'about_company', 'website', 'nse_profile', 'bse_profile', 'face_value', 'book_value', 'roce_percentage', 'roe_percentage', 'id_y', 'company_id', 'compounded_sales_growth', 'compounded_profit_growth', 'stock_price_cagr', 'roe']


In [ ]:
print(master.head())

         id_x                                       company_logo  \
0         ABB   https://mkt.in/static/mkt-icons/nifty100/ABB.png   
1  ADANIENSOL  https://m.economictimes.com/thumb/msid-1173715...   
2    ADANIENT  https://mkt.in/static/mkt-icons/nifty100/ADANI...   
3  ADANIGREEN  https://mkt.in/static/mkt-icons/nifty100/ADANI...   
4  ADANIPORTS  https://mkt.in/static/mkt-icons/nifty100/ADANI...   

                              company_name  \
0                         Abbott India Ltd   
1               Adani Energy Solutions Ltd   
2                    Adani Enterprises Ltd   
3                   Adani Green Energy Ltd   
4  Adani Ports & Special Economic Zone Ltd   

                                          chart_link  \
0  https://in.tradingview.com/chart/?symbol=NSE%3...   
1  https://in.tradingview.com/chart/?symbol=NSE%3...   
2  https://in.tradingview.com/chart/?symbol=ADANIENT   
3  https://in.tradingview.com/chart/?symbol=NSE%3...   
4  https://in.tradingview.com/char

In [ ]:
master = master.rename(columns={"id_x": "id"})

In [ ]:
print(master.columns.tolist())

['id', 'company_logo', 'company_name', 'chart_link', 'about_company', 'website', 'nse_profile', 'bse_profile', 'face_value', 'book_value', 'roce_percentage', 'roe_percentage', 'id_y', 'company_id', 'compounded_sales_growth', 'compounded_profit_growth', 'stock_price_cagr', 'roe']


In [ ]:
master = master.merge(
    latest_pl,
    left_on="id",
    right_on="company_id",
    how="left",
    suffixes=("", "_pl")
)

In [ ]:
print(peer_groups.columns.tolist())

['2', 'Private Banks', 'ICICIBANK', 'False']


In [ ]:
peer_groups = pd.read_excel("peer_groups.xlsx")

print(peer_groups.columns.tolist())

['id', 'peer_group', 'company_id', 'is_leader']


In [ ]:
master = master.merge(
    peer_groups[
        ["company_id", "peer_group", "is_leader"]
    ],
    left_on="id",
    right_on="company_id",
    how="left"
)

In [ ]:
print(master.columns.tolist())

['id', 'company_logo', 'company_name', 'chart_link', 'about_company', 'website', 'nse_profile', 'bse_profile', 'face_value', 'book_value', 'roce_percentage', 'roe_percentage', 'id_y', 'company_id_x', 'compounded_sales_growth', 'compounded_profit_growth', 'stock_price_cagr', 'roe', 'company_id_pl', 'id_pl', 'year', 'sales', 'expenses', 'operating_profit', 'opm_percentage', 'other_income', 'interest', 'depreciation', 'profit_before_tax', 'tax_percentage', 'net_profit', 'eps', 'dividend_payout', 'company_id_pl', 'id_pl', 'year_pl', 'sales_pl', 'expenses_pl', 'operating_profit_pl', 'opm_percentage_pl', 'other_income_pl', 'interest_pl', 'depreciation_pl', 'profit_before_tax_pl', 'tax_percentage_pl', 'net_profit_pl', 'eps_pl', 'dividend_payout_pl', 'company_id_y', 'peer_group', 'is_leader']


In [ ]:
companies = pd.read_excel("companies.xlsx")
analysis = pd.read_excel("analysis.xlsx")
profit_loss = pd.read_excel("profitandloss.xlsx")
peer_groups = pd.read_excel("peer_groups.xlsx")

In [ ]:
profit_loss.columns.tolist()

['id',
 'company_id',
 'year',
 'sales',
 'expenses',
 'operating_profit',
 'opm_percentage',
 'other_income',
 'interest',
 'depreciation',
 'profit_before_tax',
 'tax_percentage',
 'net_profit',
 'eps',
 'dividend_payout']

In [ ]:
latest_pl = (
    profit_loss
    .sort_values("year")
    .groupby("company_id")
    .last()
    .reset_index()
)

In [ ]:
import sqlite3

conn = sqlite3.connect("nifty100.db")

master.to_sql(
    "master_company_data",
    conn,
    if_exists="replace",
    index=False
)

conn.close()

print("✅ master_company_data updated successfully.")

✅ master_company_data updated successfully.


In [ ]:
master = companies.copy()

In [ ]:
master = master.merge(
    analysis,
    left_on="id",
    right_on="company_id",
    how="left"
)

In [ ]:
print(master.columns.tolist())

['id_x', 'company_logo', 'company_name', 'chart_link', 'about_company', 'website', 'nse_profile', 'bse_profile', 'face_value', 'book_value', 'roce_percentage', 'roe_percentage', 'id_y', 'company_id', 'compounded_sales_growth', 'compounded_profit_growth', 'stock_price_cagr', 'roe']


In [ ]:
profit_loss.columns.tolist()

['id',
 'company_id',
 'year',
 'sales',
 'expenses',
 'operating_profit',
 'opm_percentage',
 'other_income',
 'interest',
 'depreciation',
 'profit_before_tax',
 'tax_percentage',
 'net_profit',
 'eps',
 'dividend_payout']

In [ ]:
master.head()

,id_x,company_logo,company_name,chart_link,about_company,website,nse_profile,bse_profile,face_value,book_value,roce_percentage,roe_percentage,id_y,company_id,compounded_sales_growth,compounded_profit_growth,stock_price_cagr,roe
0,ABB,https://mkt.in/static/mkt-icons/nifty100/ABB.png,Abbott India Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,Abbott India Ltd is one of the leading multina...,https://www.abbott.co.in/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/abb...,10.0,1657.0,46.0,34.90,NaN,NaN,NaN,NaN,NaN,NaN
1,ADANIENSOL,https://m.economictimes.com/thumb/msid-1173715...,Adani Energy Solutions Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"AESL, part of the Adani portfolio, is a multid...",https://www.adanienergysolutions.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10.0,175.0,9.0,8.59,NaN,NaN,NaN,NaN,NaN,NaN
2,ADANIENT,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Enterprises Ltd,https://in.tradingview.com/chart/?symbol=ADANIENT,Adani Enterprises Ltd is an Indian multination...,https://www.adanienterprises.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,1.0,363.0,11.6,13.64,NaN,NaN,NaN,NaN,NaN,NaN
3,ADANIGREEN,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Green Energy Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"Adani Green Energy Limited, incorporated in 20...",http://www.adanigreenenergy.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10.0,67.0,96.5,14.70,NaN,NaN,NaN,NaN,NaN,NaN
4,ADANIPORTS,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Ports & Special Economic Zone Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,Adani Ports & Special Economic Zone is in the ...,http://www.adaniports.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,2.0,265.0,12.9,18.10,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
master = master.rename(columns={"id_x": "id"})

In [ ]:
master = master.merge(
    latest_pl,
    left_on="id",
    right_on="company_id",
    how="left",
    suffixes=("", "_pl")
)

In [ ]:
latest_pl = (
    profit_loss
    .sort_values("year")
    .groupby("company_id")
    .last()
    .reset_index()
)

In [ ]:
master = master.merge(
    latest_pl,
    left_on="id",
    right_on="company_id",
    how="left",
    suffixes=("", "_pl")
)

In [ ]:
print(master.columns[master.columns.duplicated()])

Index(['company_id_pl', 'id_pl'], dtype='object')


In [ ]:
master.columns.tolist()

['id',
 'company_logo',
 'company_name',
 'chart_link',
 'about_company',
 'website',
 'nse_profile',
 'bse_profile',
 'face_value',
 'book_value',
 'roce_percentage',
 'roe_percentage',
 'id_y',
 'company_id',
 'compounded_sales_growth',
 'compounded_profit_growth',
 'stock_price_cagr',
 'roe',
 'company_id_pl',
 'id_pl',
 'year',
 'sales',
 'expenses',
 'operating_profit',
 'opm_percentage',
 'other_income',
 'interest',
 'depreciation',
 'profit_before_tax',
 'tax_percentage',
 'net_profit',
 'eps',
 'dividend_payout',
 'company_id_pl',
 'id_pl',
 'year_pl',
 'sales_pl',
 'expenses_pl',
 'operating_profit_pl',
 'opm_percentage_pl',
 'other_income_pl',
 'interest_pl',
 'depreciation_pl',
 'profit_before_tax_pl',
 'tax_percentage_pl',
 'net_profit_pl',
 'eps_pl',
 'dividend_payout_pl']

In [ ]:
master.columns[master.columns.duplicated()]

Index(['company_id_pl', 'id_pl'], dtype='object')

In [ ]:
master = master.loc[:, ~master.columns.duplicated()]

In [ ]:
master.columns.tolist()

['id',
 'company_logo',
 'company_name',
 'chart_link',
 'about_company',
 'website',
 'nse_profile',
 'bse_profile',
 'face_value',
 'book_value',
 'roce_percentage',
 'roe_percentage',
 'id_y',
 'company_id',
 'compounded_sales_growth',
 'compounded_profit_growth',
 'stock_price_cagr',
 'roe',
 'company_id_pl',
 'id_pl',
 'year',
 'sales',
 'expenses',
 'operating_profit',
 'opm_percentage',
 'other_income',
 'interest',
 'depreciation',
 'profit_before_tax',
 'tax_percentage',
 'net_profit',
 'eps',
 'dividend_payout',
 'year_pl',
 'sales_pl',
 'expenses_pl',
 'operating_profit_pl',
 'opm_percentage_pl',
 'other_income_pl',
 'interest_pl',
 'depreciation_pl',
 'profit_before_tax_pl',
 'tax_percentage_pl',
 'net_profit_pl',
 'eps_pl',
 'dividend_payout_pl']

In [ ]:
import sqlite3

conn = sqlite3.connect("nifty100.db")

master.to_sql(
    "master_company_data",
    conn,
    if_exists="replace",
    index=False
)

conn.close()

print("Database Updated")

Database Updated


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("nifty100.db")

tables = pd.read_sql(
"""
SELECT name
FROM sqlite_master
WHERE type='table'
""",
conn)

tables

,name
0,master_company_data
1,companies
2,sectors
3,peer_groups
4,profit_loss
5,balance_sheet
6,cash_flow
7,financial_ratios
8,market_cap
9,stock_prices
